# Mini2P single-run QC: bidiphase, destriping, distortion, Suite2p registration

This notebook inspects one recording. All processing lives in the `mini2p_preproc` package; the cells here only call it and draw figures, so what you see is exactly what the batch runs produce.

Order of operations:

1. Estimate one odd/even-line scan-phase offset per chunk of `phase.chunk_frames` timepoints and apply it to both channels (optionally a second residual pass).
2. Optionally destripe (`destripe.enabled`), preserving each frame's mean fluorescence.
3. Distortion-correct with the fixed calibration (`distortion.mode`).
4. Register with Suite2p, with Suite2p's own bidiphase correction disabled.

Settings and the recording to inspect come from `config/pipeline.toml` (copy `config/pipeline.example.toml`). Change settings there, not here, so the notebook and batch runs always agree.

In [ ]:
import logging
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display

from mini2p_preproc import plots, qc
from mini2p_preproc.bidiphase import adjacent_line_score, apply_bidiphase, select_alignment
from mini2p_preproc.config import (
    corrected_tiff_path,
    load_qc_run,
    load_settings,
    run_name_from_output_dir,
    variant_output_dir,
)
from mini2p_preproc.contrast import scale_contrast
from mini2p_preproc.correction import correction_stages, preprocess_tiff
from mini2p_preproc.distortion import write_distortion_corrected
from mini2p_preproc.exports import Contrast, save_contrast_choice
from mini2p_preproc.pipeline import suite2p_input_for
from mini2p_preproc.registration import load_registered_outputs, run_registration

logging.basicConfig(level=logging.INFO, format="%(name)s: %(message)s", force=True)

CONFIG = Path("../config/pipeline.toml")
settings = load_settings(CONFIG)
run = load_qc_run(CONFIG)
OUTPUT_DIR = variant_output_dir(run.run_dir, settings.destripe.enabled)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CORRECTED_TIFF = corrected_tiff_path(OUTPUT_DIR, run.input_tiff, settings)
CHANNEL_NAMES = settings.channels.names

print("Input:", run.input_tiff)
print("Output folder:", OUTPUT_DIR)
print("Destriping:", settings.destripe.enabled)
print("Phase source:", settings.phase.alignment_source,
      "| motion source:", settings.registration.motion_alignment_source)

## Preview phase estimates and search quality

Run this before writing the full TIFF. Top: phase per chunk (ringed points are at or near the search boundary). Bottom: best normalized adjacent-line score per channel.

- Values reaching ±`max_shift_px`: increase the search range.
- Estimates jumping within a section whose mean stays doubled: reduce `chunk_frames`.
- The score curve for the lowest-scoring chunk reveals ambiguous competing peaks.

In [ ]:
trace = qc.phase_trace(run.input_tiff, settings)
plots.plot_phase_trace(trace, settings)
plt.show()

phases = [row["phase_px"] for row in trace]
print("Phase range:", (min(phases), max(phases)))
boundary_starts = [row["start"] for row in trace if row["at_boundary"]]
if boundary_starts:
    print("WARNING: search-boundary chunks start at", boundary_starts,
          "- increase phase.max_shift_px and rerun.")

worst = qc.lowest_score_chunk(trace)
worst_chunks = qc.load_chunk(run.input_tiff, worst["start"], worst["end"], settings.channels.n_channels)
if settings.phase.estimator == "wide":
    worst_choice = select_alignment(worst_chunks, settings.phase, settings.channels)
    plots.plot_score_curve(
        worst_choice.estimate,
        f"Lowest-score chunk {worst['start']}:{worst['end']} ({worst_choice.selected_source})",
    )
    plt.show()

## Manual offset for the lowest-scoring chunk

The slider applies an integer offset to both channels of the chunk above and compares it with the uncorrected mean. Edge columns affected by the largest shift are hidden. When it looks right, copy the printed line into `[phase.manual_first_phase_by_start]` in the config. Nothing is written here.

In [ ]:
raw_means = qc.mean_images(worst_chunks)
crop = settings.phase.max_shift_px
x_slice = slice(crop, -crop) if 2 * crop < raw_means[0].shape[1] else slice(None)
score_channel = worst["selected_channel_zero_based"]


def show_offset(offset_px):
    corrected_means = qc.mean_images([apply_bidiphase(c, offset_px) for c in worst_chunks])
    axes = plots.plot_channel_pairs(
        raw_means, corrected_means, ("Uncorrected", f"Manual offset {offset_px:+d} px"),
        CHANNEL_NAMES, x_slice=x_slice,
    )
    score = adjacent_line_score(corrected_means[score_channel], crop)
    axes[0, 0].figure.suptitle(
        f"Chunk {worst['start']}:{worst['end']} — score {score:.4f} "
        f"(automatic {worst['phase_px']:+d} px)", fontsize=14,
    )
    plt.show()
    print(f"Config line under [phase.manual_first_phase_by_start]:  {worst['start']} = {offset_px}")


slider = widgets.IntSlider(value=worst["phase_px"], min=-crop, max=crop, step=1,
                           description="Offset (px):", continuous_update=False,
                           layout=widgets.Layout(width="650px"))
display(slider, widgets.interactive_output(show_offset, {"offset_px": slider}))

## Write the corrected TIFF

Writes `<stem>_dynamic_bidiphase[_destriped].tif` plus a `.phase.json` sidecar recording every chunk's phases and the settings. With `overwrite = false` an existing file is never replaced.

In [ ]:
correction = preprocess_tiff(
    run.input_tiff, CORRECTED_TIFF, settings, run_name_from_output_dir(OUTPUT_DIR)
)
print("Whole-recording second phase:", correction.global_second_phase_px)

## QC 1, 1b, 2 — each correction stage for one chunk

Uses exactly the function that wrote the TIFF, so when `qc_start_timepoint` is a multiple of `phase.chunk_frames` these are the written pixels. Each row is one channel; both columns share display limits.

In [ ]:
qc_start = run.qc_start_timepoint
qc_raw = qc.load_chunk(run.input_tiff, qc_start, qc_start + settings.phase.chunk_frames,
                       settings.channels.n_channels)
stages = correction_stages(qc_raw, qc_start, settings, correction.global_residual)
print(f"Timepoints {qc_start}:{qc_start + len(qc_raw[0])}; first={stages.first_phase_px:+d} px; "
      f"second={stages.second_phase_px:+d} px; source={stages.first_choice.selected_source}; "
      f"scores={stages.first_choice.score_by_source}")

plots.plot_channel_pairs(qc.mean_images(qc_raw), qc.mean_images(stages.after_first),
                         ("Before", "After first bidiphase"), CHANNEL_NAMES)
plt.show()
if settings.phase.run_second_pass:
    plots.plot_channel_pairs(qc.mean_images(stages.after_first), qc.mean_images(stages.after_second),
                             ("After first bidiphase", f"After second ({settings.phase.second_pass_mode})"),
                             CHANNEL_NAMES)
    plt.show()
if settings.destripe.enabled:
    plots.plot_channel_pairs(qc.mean_images(stages.after_second), qc.mean_images(stages.corrected),
                             ("Before destriping", "After destriping"), CHANNEL_NAMES)
    plt.show()

## Distortion correction (before registration)

`local_batched` applies one fixed calibration transform to every frame, `distortion.batch_frames` timepoints at a time; the transform and crop are computed once for the recording, so batch size never changes the pixels. `cluster_full` expects the SLURM worker's output to exist already; `off` registers the corrected TIFF directly.

In [ ]:
SUITE2P_INPUT = suite2p_input_for(OUTPUT_DIR, run.input_tiff, settings)
if settings.distortion.mode == "local_batched":
    write_distortion_corrected(CORRECTED_TIFF, SUITE2P_INPUT, settings)
elif settings.distortion.mode == "cluster_full" and not SUITE2P_INPUT.is_file():
    raise FileNotFoundError(f"Cluster distortion output not found: {SUITE2P_INPUT}")
print("Suite2p input:", SUITE2P_INPUT)

if settings.distortion.mode != "off":
    n = run.qc_distortion_frames
    before = qc.mean_images(qc.load_chunk(CORRECTED_TIFF, 0, n, settings.channels.n_channels))
    after = qc.mean_images(qc.load_chunk(SUITE2P_INPUT, 0, n, settings.channels.n_channels))
    plots.plot_channel_pairs(before, after, ("Before distortion correction", "After"), CHANNEL_NAMES)
    plt.show()
    print("Distortion output shape:", after[0].shape)

## Suite2p registration

Suite2p's `do_bidiphase` is off because phase was corrected per chunk above. `registration.motion_alignment_source` picks the channel that supplies motion shifts; they are applied to both channels. Green stays the functional channel.

In [ ]:
PLANE_DIR = run_registration(SUITE2P_INPUT, OUTPUT_DIR, settings)
registered = load_registered_outputs(PLANE_DIR)
print("Suite2p output:", PLANE_DIR)

## QC 4 and registration diagnostics

Before/after means over the same first frames; then rigid offsets, frame-to-reference correlation, bad frames, nonrigid offsets, and the first-N registered mean against Suite2p's recording-wide mean. Large offset jumps at phase-chunk boundaries suggest reducing `chunk_frames`.

In [ ]:
n = run.qc_registration_frames
before_registration = qc.mean_images(qc.load_chunk(SUITE2P_INPUT, 0, n, settings.channels.n_channels))
first_n_registered = qc.registered_first_n_means(registered, n, settings)
plots.plot_channel_pairs(before_registration, first_n_registered,
                         ("Before Suite2p registration", "After"), CHANNEL_NAMES)
plt.show()

ops = registered.ops
for key in ["fs", "nframes", "Ly", "Lx", "nchannels", "functional_chan", "align_by_chan2",
            "do_bidiphase", "bidiphase", "nonrigid", "block_size", "maxregshift", "maxregshiftNR",
            "smooth_sigma", "smooth_sigma_time", "nimg_init", "batch_size",
            "two_step_registration", "norm_frames", "th_badframes", "snr_thresh"]:
    if key in ops:
        print(f"{key:>24}: {ops[key]}")

plots.plot_registration_traces(ops, settings.registration.fs_hz)
plt.show()
print(f"Bad frames: {int(np.sum(ops.get('badframes', [])))}/{len(ops.get('xoff', []))}")
if np.ndim(ops.get("xoff1", [])) >= 2:
    plots.plot_nonrigid_offsets(ops, settings.registration.fs_hz)
    plt.show()

functional_mean, structural_mean = qc.suite2p_mean_images(ops)
by_channel = {settings.channels.functional_channel: functional_mean,
              settings.channels.structural_channel: structural_mean}
plots.plot_channel_pairs(first_n_registered,
                         [by_channel[ch] for ch in range(1, settings.channels.n_channels + 1)],
                         (f"First {n} registered frames", "Suite2p recording-wide mean"),
                         CHANNEL_NAMES)
plt.show()

## Registered mean images: contrast and overlay

Contrast controls affect display only. The next cell saves your chosen contrast beside `ops.npy`.

In [ ]:
def contrast_sliders(prefix):
    return (
        widgets.FloatSlider(value=1.0, min=0.0, max=50.0, step=0.5, description=f"{prefix} black %",
                            continuous_update=False),
        widgets.FloatSlider(value=99.8, min=90.0, max=100.0, step=0.05,
                            description=f"{prefix} white %", continuous_update=False),
        widgets.FloatSlider(value=1.0, min=0.2, max=3.0, step=0.05, description=f"{prefix} gamma",
                            continuous_update=False),
    )


green_sliders, red_sliders = contrast_sliders("Green"), contrast_sliders("Red")


def preview(gb, gw, gg, rb, rw, rg):
    if gw <= gb or rw <= rb:
        print("Each white percentile must be above its black percentile")
        return
    green, *green_range = scale_contrast(functional_mean, gb, gw, gg)
    red, *red_range = scale_contrast(structural_mean, rb, rw, rg)
    plots.plot_contrast_panels(green, red, green_range, red_range)
    plt.show()


keys = ["gb", "gw", "gg", "rb", "rw", "rg"]
display(widgets.HBox([widgets.VBox(green_sliders), widgets.VBox(red_sliders)]),
        widgets.interactive_output(preview, dict(zip(keys, (*green_sliders, *red_sliders)))))

In [ ]:
chosen = [Contrast(*(s.value for s in sliders)) for sliders in (green_sliders, red_sliders)]
for label, path in save_contrast_choice(PLANE_DIR, functional_mean, structural_mean, *chosen).items():
    print(f"Saved {label}: {path}")